## Week 4 : Data Cleaning and Preparation

In [1]:
# import libraries
import pandas as pd

In [2]:
# load file
listing = pd.read_csv("/Users/yoorachoi/Python/IDX/Listing/data/CRMLSListing_with_rates.csv", low_memory=False)
sold = pd.read_csv("/Users/yoorachoi/Python/IDX/Sold/CRMLSSold_with_rates.csv", low_memory=False)

### 1. Convert date fields to datetime format 

In [65]:
# Convert date fields to datetime format
def convert_date_colums(df):
    date_cols = [col for col in df.columns if 'date' in col.lower()]
    print(f"Total {len(date_cols)} date fields : {date_cols}")

    
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors = 'coerce')
        print (f"{col} : successfully converted to datetime")
      
    print()
    return df
    

In [66]:
# Run sold & listing : Convert date fields to datetime format in sold & listing data

sold = convert_date_colums(sold)
listing = convert_date_colums(listing)

Total 4 date fields : ['CloseDate', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate']
CloseDate : successfully converted to datetime
ContractStatusChangeDate : successfully converted to datetime
PurchaseContractDate : successfully converted to datetime
ListingContractDate : successfully converted to datetime

Total 5 date fields : ['CloseDate', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'CloseDate.1']
CloseDate : successfully converted to datetime
ContractStatusChangeDate : successfully converted to datetime
PurchaseContractDate : successfully converted to datetime
ListingContractDate : successfully converted to datetime
CloseDate.1 : successfully converted to datetime



### 2-1. Remove unnecessary or redundant columns

In [67]:
# check duplicate columns
pairs = []

for col in listing.columns:
    if '.' in col:
        base_col = col.split('.')[0]
        if base_col in listing.columns:
            pairs.append((base_col, col))

pairs

[('Longitude', 'Longitude.1'),
 ('Latitude', 'Latitude.1'),
 ('CloseDate', 'CloseDate.1'),
 ('BuyerOfficeName', 'BuyerOfficeName.1')]

In [68]:
# Check if duplicate column pairs have identical values row-by-row
for base, dup in pairs:
    same_ratio = (listing[base] == listing[dup]).mean()
    print(f"{base} vs {dup}: {same_ratio:.2f}")

Longitude vs Longitude.1: 1.00
Latitude vs Latitude.1: 1.00
CloseDate vs CloseDate.1: 0.31
BuyerOfficeName vs BuyerOfficeName.1: 1.00


In [69]:
# Remove unnecessary or redundant columns: Detect columns ending with '.1' as duplicates and drop them if they are more than 90% identical to the base column.
    
def drop_duplicate_columns(df, threshold=0.9):
    
    # Auto-detect columns ending with '.1'
    dup_cols = [col for col in df.columns if col.endswith('.1')]
    pairs = [(col.replace('.1', ''), col) for col in dup_cols]
    
    print(f"Duplicate pairs found ({len(pairs)}): {pairs}")
    
    cols_to_drop = []
    
    for base, dup in pairs:
        if base in df.columns and dup in df.columns:
            same_ratio = (df[base] == df[dup]).mean()
            
            if same_ratio >= threshold:
                print(f"DROP COMPELETED {base} vs {dup}: {same_ratio:.2f} → dropping '{dup}'")
                cols_to_drop.append(dup)
            else:
                print(f"DROP FAILED {base} vs {dup}: {same_ratio:.2f} → kept (values differ)")
    
    df = df.drop(columns=cols_to_drop)
    
    print(f"\n Columns dropped: {len(cols_to_drop)}")
    print(f"   Final column count: {df.shape[1]}")
    
    return df


In [47]:
# Run sold & listing : Remove unnecessary or redundant columns
sold = drop_duplicate_columns(sold)


Duplicate pairs found (0): []

 Columns dropped: 0
   Final column count: 69


In [48]:
listing = drop_duplicate_columns(listing)

Duplicate pairs found (11): [('PropertyType', 'PropertyType.1'), ('ListAgentFirstName', 'ListAgentFirstName.1'), ('DaysOnMarket', 'DaysOnMarket.1'), ('LivingArea', 'LivingArea.1'), ('Longitude', 'Longitude.1'), ('Latitude', 'Latitude.1'), ('ListPrice', 'ListPrice.1'), ('ListAgentLastName', 'ListAgentLastName.1'), ('CloseDate', 'CloseDate.1'), ('BuyerOfficeName', 'BuyerOfficeName.1'), ('UnparsedAddress', 'UnparsedAddress.1')]
DROP COMPELETED PropertyType vs PropertyType.1: 1.00 → dropping 'PropertyType.1'
DROP COMPELETED ListAgentFirstName vs ListAgentFirstName.1: 0.99 → dropping 'ListAgentFirstName.1'
DROP COMPELETED DaysOnMarket vs DaysOnMarket.1: 1.00 → dropping 'DaysOnMarket.1'
DROP COMPELETED LivingArea vs LivingArea.1: 1.00 → dropping 'LivingArea.1'
DROP FAILED Longitude vs Longitude.1: 0.85 → kept (values differ)
DROP FAILED Latitude vs Latitude.1: 0.85 → kept (values differ)
DROP COMPELETED ListPrice vs ListPrice.1: 1.00 → dropping 'ListPrice.1'
DROP COMPELETED ListAgentLastName

### 2-2. Remove duplicate rows

In [51]:
# Remove duplicate rows and print a summary.
def drop_duplicate_rows(df, dataset_name='dataset'):
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    
    print(f"  [{dataset_name}]")
    print(f"   Before : {before:,} rows")
    print(f"   After  : {after:,} rows")
    print(f"   Removed: {before - after:,} duplicate rows")
    
    return df


In [52]:
# Run sold & listing : Remove duplicate rows
sold = drop_duplicate_rows(sold, dataset_name='Sold')


  [Sold]
   Before : 397,603 rows
   After  : 397,555 rows
   Removed: 48 duplicate rows


In [54]:
listing = drop_duplicate_rows(listing, dataset_name='Listing')

  [Listing]
   Before : 540,183 rows
   After  : 540,183 rows
   Removed: 0 duplicate rows


### 3. Handle missing values appropriately by data type
- numeric column : Based on EDA, all numerical variables tend to be light-skewed, so I replace null values in numeric columns to median.
- categorical column : replace null to Unknown
- Datetime column : keep null 

In [57]:
def handle_missing_values(df):
    
    numeric_cols     = df.select_dtypes(include='number').columns.tolist()
    categorical_cols = df.select_dtypes(include='object').columns.tolist()
    datetime_cols    = df.select_dtypes(include='datetime').columns.tolist()

    # Numeric → median
    for col in numeric_cols:
        missing = df[col].isnull().sum()
        if missing > 0:
            df[col] = df[col].fillna(df[col].median())
            print(f" {col}: {missing} missing → filled with median")

    # Categorical → 'Unknown'
    for col in categorical_cols:
        missing = df[col].isnull().sum()
        if missing > 0:
            df[col] = df[col].fillna('Unknown')
            print(f"{col}: {missing} missing → filled with 'Unknown'")

    # Datetime → keep NaT
    for col in datetime_cols:
        missing = df[col].isnull().sum()
        if missing > 0:
            print(f"{col}: {missing} missing → kept as NaT")

    print(f"\nMissing value handling complete!")
    print()
    return df


In [56]:
# Run
sold     = handle_missing_values(sold)
listing = handle_missing_values(listing)

ContractStatusChangeDate: 589 missing → kept as NaT
PurchaseContractDate: 194 missing → kept as NaT
ListingContractDate: 1 missing → kept as NaT

Missing value handling complete!

 OriginalListPrice: 774 missing → filled with median
 ClosePrice: 394603 missing → filled with median
 Latitude: 80145 missing → filled with median
 Longitude: 80145 missing → filled with median
 LivingArea: 556 missing → filled with median
 ParkingTotal: 20 missing → filled with median
 LotSizeAcres: 44518 missing → filled with median
 YearBuilt: 939 missing → filled with median
 StreetNumberNumeric: 850 missing → filled with median
 BathroomsTotalInteger: 55 missing → filled with median
 BuyerAgencyCompensation: 461190 missing → filled with median
 BedroomsTotal: 148 missing → filled with median
 Longitude.1: 80145 missing → filled with median
 Latitude.1: 80145 missing → filled with median
 Stories: 94249 missing → filled with median
 LotSizeArea: 43679 missing → filled with median
 MainLevelBedrooms: 2460

### 4. Ensure numeric fields are properly typed 

In [71]:
# Check sold data
print(sold.select_dtypes(include='object').dtypes)

UnparsedAddress                object
PropertyType                   object
ListOfficeName                 object
BuyerOfficeName                object
CoListOfficeName               object
ListAgentFullName              object
CoListAgentFirstName           object
CoListAgentLastName            object
BuyerAgentMlsId                object
BuyerAgentFirstName            object
BuyerAgentLastName             object
AssociationFeeFrequency        object
MLSAreaMajor                   object
CountyOrParish                 object
MlsStatus                      object
ElementarySchool               object
AttachedGarageYN               object
PropertySubType                object
SubdivisionName                object
BuyerOfficeAOR                 object
ListingId                      object
City                           object
StateOrProvince                object
MiddleOrJuniorSchool           object
FireplaceYN                    object
HighSchool                     object
Levels      

In [72]:
# Check listing data
print(listing.select_dtypes(include='object').dtypes)

ListAgentEmail                 object
ListAgentFirstName             object
ListAgentLastName              object
UnparsedAddress                object
PropertyType                   object
ListOfficeName                 object
BuyerOfficeName                object
CoListOfficeName               object
ListAgentFullName              object
CoListAgentFirstName           object
CoListAgentLastName            object
BuyerAgentMlsId                object
BuyerAgentFirstName            object
BuyerAgentLastName             object
AssociationFeeFrequency        object
MLSAreaMajor                   object
CountyOrParish                 object
MlsStatus                      object
ElementarySchool               object
AttachedGarageYN               object
PropertySubType                object
SubdivisionName                object
BuyerOfficeAOR                 object
BuyerAgencyCompensationType    object
ListingId                      object
City                           object
StateOrProvi

### 4. Remove or flag invalid numeric values (Outliers)
ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, negative Bedrooms or Bathrooms